# Linear SVM

Le Support Vector Machines sono classificatori lineari che ricercano iperpiani di separazione con margine massimo.

L'obiettivo (primale) dell'SVM consiste nella minimizzazione di:

$$J(w, b) = \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\max(0, 1 - z_i(w^T x_i + b))$$

dove $n$ è il numero di campioni di addestramento, $C$ è un iper-parametro e $z_i$ è l'etichetta di classe per l'$i$-esimo campione, codificata come:

$$z_i = \begin{cases} +1 & \text{se } x_i \text{ appartiene alla classe HT} \\ -1 & \text{se } x_i \text{ appartiene alla classe HF} \end{cases}$$

Come abbiamo visto, per risolvere il problema SVM possiamo considerare anche la formulazione duale:

$$J_D(\alpha) = -\frac{1}{2}\alpha^T H\alpha + \alpha^T \mathbf{1}$$

$$\text{s.t. } 0 \leq \alpha_i \leq C, \quad \forall i \in \{1 \ldots n\}, \quad \sum_{i=1}^{n}\alpha_i z_i = 0$$

dove $\mathbf{1}$ è un vettore $n$-dimensionale di uni, e $H$ è una matrice i cui elementi sono:

$$H_{ij} = z_i z_j x_i^T x_j$$

La soluzione duale dell'SVM è il massimizzatore di $J_D(\alpha)$. Le soluzioni duale e primale $\alpha^*$ e $w^*$ sono correlate attraverso:

$$w^* = \sum_{i=1}^{n}\alpha_i^* z_i x_i$$

e il bias ottimale $b^*$ può essere calcolato considerando un campione $x_i$ che giace sul margine:

$$z_i(w^{*T}x_i + b^*) = 1$$

e quindi risolvendo per $b^*$. Come discusso, il problema duale permette anche la separazione non-lineare attraverso il kernel trick. Infatti, possiamo sostituire i prodotti scalari $x_i^T x_j$ con funzioni kernel $k(x_i, x_j)$, in modo che $H_{ij} = z_i z_j k(x_i, x_j)$, e possiamo calcolare $w^{*T}x$ attraverso funzioni kernel:

$$w^{*T}x = \sum_{i=1}^{n}\alpha_i^* z_i k(x_i, x) = \sum_{i|\alpha_i \neq 0}\alpha_i^* z_i k(x_i, x)$$

Per gli SVM lineari, possiamo ottimizzare sia la formulazione primale che quella duale. Sfortunatamente, la formulazione non vincolata della funzione obiettivo primale non è differenziabile. Sebbene il metodo L-BFGS possa comunque trovare il minimizzatore di $J$, non abbiamo alcuna garanzia che l'algoritmo si fermi vicino al valore ottimale di $(w, b)$. Risolutori numerici ad-hoc sono stati sviluppati, tuttavia sono al di fuori dello scopo del nostro corso.

La formulazione duale è differenziabile, tuttavia contiene sia vincoli di scatola (cioè vincoli della forma $a \leq \alpha_i \leq b$) che il vincolo aggiuntivo $\sum_{i=1}^{n}\alpha_i z_i = 0$. L'algoritmo L-BFGS che abbiamo impiegato è in grado di gestire i vincoli di scatola, tuttavia non può incorporare quest'ultimo.

Il vincolo $\sum_{i=1}^{n}\alpha_i z_i = 0$ deriva dalla presenza del termine bias nella formulazione primale dell'SVM. Pertanto, modifichiamo leggermente il problema SVM in modo da far scomparire il vincolo. In questo modo saremo in grado di impiegare L-BFGS-B (la B sta per box-constraints) per risolvere il problema duale.

Riformuliamo il problema primale come minimizzazione di:

$$J_b(w_b) = \frac{1}{2}\|w_b\|^2 + C\sum_{i=1}^{n}\max(0, 1 - z_i(w_b^T x_b^i))$$

dove:

$$x_b^i = \begin{pmatrix} x_i \\ 1 \end{pmatrix}, \quad w_b = \begin{pmatrix} w \\ b \end{pmatrix}$$

Possiamo osservare che $w_b^T x_b^i = w^T x + b$, cioè le regole di scoring hanno la stessa espressione della formulazione originale. Tuttavia, a differenza del problema SVM originale, stiamo anche regolarizzando il valore del termine bias, poiché regolarizziamo la norma di $w_b$:

$$\|w_b\|^2 = \|w\|^2 + b^2$$

La regolarizzazione del bias può, in generale, portare a decisioni sub-ottimali in termini di margine di separazione. Possiamo mitigare questo effetto utilizzando una mappatura:

$$x_b^i = \begin{pmatrix} x_i \\ K \end{pmatrix}$$

Man mano che $K$ diventa più grande, gli effetti della regolarizzazione di $b$ diventano più deboli, tuttavia l'algoritmo potrebbe richiedere più iterazioni per convergere a una soluzione.

L'obiettivo duale del problema SVM primale modificato diventa la massimizzazione di:

$$J_b^D(\alpha) = -\frac{1}{2}\alpha^T H_c\alpha + \alpha^T \mathbf{1}$$

$$\text{s.t. } 0 \leq \alpha_i \leq C, \quad \forall i \in \{1 \ldots n\}$$

cioè, la stessa formulazione di prima ma senza il vincolo di uguaglianza e con la matrice $H_c$ calcolata dalle caratteristiche estese $x_b^i$ anziché dalle caratteristiche originali $x_i$:

$$H_c^{i,j} = z_i z_j x_b^{i\,T} x_b^j$$

Scrivere una funzione che calcola la soluzione SVM primale attraverso la formulazione SVM duale $J_b^D$.

In [120]:
import numpy as np
import scipy.optimize 
import sklearn

def load_iris_binary():
    D, L = sklearn.datasets.load_iris()['data'].T, sklearn.datasets.load_iris()['target']
    D = D[:, L != 0] # We remove setosa from D
    L = L[L!=0] # We remove setosa from L
    L[L==2] = 0 # We assign label 0 to virginica (was label 2)
    return D, L
def split_db_2to1(D, L, seed=0):
    nTrain = int(D.shape[1]*2.0/3.0)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    idxTrain = idx[0:nTrain]
    idxTest = idx[nTrain:]
    DTR = D[:, idxTrain]
    DVAL = D[:, idxTest]
    LTR = L[idxTrain]
    LVAL = L[idxTest]
    return (DTR, LTR), (DVAL, LVAL)

D, L = load_iris_binary()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)


In [121]:
def getMu(D):
    return D.sum(axis=1)/D.shape[1]

def getRow(vet):
    return vet.reshape((1, vet.size))
def getCol(vet):
     return vet.reshape((vet.size, 1))

def getC(D):
    mu=getMu(D)
    mu=getCol(mu)
    Dc=D-mu
    return (Dc @ Dc.T)/Dc.shape[1]

In [122]:
def train_SVM(K, C, DTR, LTR):
    N = DTR.shape[1]
    ones = np.ones((1, N))
    D = np.vstack([DTR, K * ones])
    G = D.T @ D

    ZTR = LTR * 2 - 1
    H = getCol(ZTR) * getRow(ZTR) * G

    def fOpt(alpha):
        Ha = H @ alpha
        loss = 0.5 * alpha.T @ Ha - alpha.sum()
        grad = Ha - np.ones(alpha.size)
        return loss, grad

    x0 = np.zeros(N)
    alpha, _, _ = scipy.optimize.fmin_l_bfgs_b(
        func=fOpt,
        x0=x0,
        approx_grad=False,
        bounds=[(0, C) for _ in range(N)],
        factr=np.nan,
        pgtol=1e-5
    )
    wb = D @ (alpha * ZTR)
    w = wb[:-1]
    b = wb[-1] * K
    
    dual_loss,_=fOpt(alpha)
    val=1-(ZTR*(wb.T @ D))
    primal_loss=1/2*(wb.T @ wb) + C*np.where(val>0,val,0).sum()
    print(f"Primal Loss : {primal_loss} - Dual Loss : {-dual_loss}")
    return alpha,w, b

alpha,w, b = train_SVM(1, 0.1, DTR, LTR) #esempio di funzionamento

Primal Loss : 3.7757815820941127 - Dual Loss : 3.774940483626768


In [123]:
def get_confusion_mat(predictions,L):
    labels=np.unique(L)
    M=np.zeros((len(labels),len(labels)))

    predictions_int = np.array(predictions).astype(int)
    L_int = np.array(L).astype(int)

    for i in range(len(predictions_int)):
        M[predictions_int[i]][L_int[i]]+=1
    
    return M
def DCF_u(predictions,L,prior,Cfn,Cfp):
    C=np.array([[0,Cfn],[Cfp,0]])
    M=get_confusion_mat(predictions,L)
    cols_sum=np.sum(M,axis=0)
    R=M/cols_sum 
    expected_cost_per_class=np.sum(R*C,axis=0)
    prior_array = np.array([1 - prior, prior])
    DCFu=np.sum(expected_cost_per_class * prior_array)
    return DCFu

def DCF_norm(predictions,L,prior,Cfn,Cfp):
    B_dummy=min(prior*Cfn,(1-prior)*Cfp)
    dcf_u=DCF_u(predictions,L,prior,Cfn,Cfp)
    return dcf_u/B_dummy
def get_bayes_decision(llr,pi,Cfn,Cfp):
    pi_Ht=pi
    pi_Hf=1-pi
    t = -np.log((pi_Ht * Cfn) / (pi_Hf * Cfp))
    predictions=np.where(llr>t,1,0)

    return predictions

def compute_min_dcf(llr, L, pi, Cfn, Cfp):
    # Ordinamento degli score (LLR) per ricavare s_1...s_M
    sorted_llr = np.sort(llr)
    
    # Creazione del vettore delle soglie includendo gli estremi -inf e +inf
    thresholds = np.concatenate([np.array([-np.inf]), sorted_llr, np.array([np.inf])])
    
    # Inizializzazione del minimo a un valore infinitamente grande
    min_dcf = np.inf
    
    # Iterazione su tutte le possibili soglie t
    for t in thresholds:
        # Calcolo delle predizioni basate sulla soglia corrente
        predictions = np.where(llr > t, 1, 0)

        # Calcolo della DCF normalizzata
        dcf_n = DCF_norm(predictions,L,pi,Cfn,Cfp)
        
        # Aggiornamento del valore minimo se la soglia corrente risulta più performante
        if dcf_n < min_dcf:
            min_dcf = dcf_n
            
    return min_dcf

In [124]:
KC = np.array([
    [1.0, 0.1],
    [1.0, 1.0],
    [1.0, 10.0],
    [10.0, 0.1],
    [10.0, 1.0],
    [10.0, 10.0]
])

pi_T=0.5
for kc in KC:
    K=kc[0]
    C=kc[1]

    print(f"================================= K: {K} C: {C}==============================================================")
    alpha,w,b=train_SVM(K,C,DTR,LTR)
    s=w.T @ DVAL + b
    predictions=s >0
    error_rate=np.mean(predictions != LVAL)
    min_dcf=compute_min_dcf(s,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(s, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)


    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

    print(f"==================================================================================================")



================================= K: 1.0 C: 0.1==============================================================
Primal Loss : 3.7757815820941127 - Dual Loss : 3.774940483626768
Error Rate: 2.9%
DCF min: 0.05555555555555555
Actual DCF:0.0625

================================= K: 1.0 C: 1.0==============================================================
Primal Loss : 15.77993089028593 - Dual Loss : 15.779930849659776
Error Rate: 5.9%
DCF min: 0.05555555555555555
Actual DCF:0.11805555555555555

================================= K: 1.0 C: 10.0==============================================================
Primal Loss : 78.96892964973537 - Dual Loss : 78.96892834899842
Error Rate: 5.9%
DCF min: 0.05555555555555555
Actual DCF:0.11805555555555555

================================= K: 10.0 C: 0.1==============================================================
Primal Loss : 2.9835765069932436 - Dual Loss : 2.983576258458373
Error Rate: 11.8%
DCF min: 0.0
Actual DCF:0.2222222222222222

================

# Kernel SVM

Le SVM permettono la classificazione non-lineare attraverso un'espansione implicita delle caratteristiche in uno spazio a dimensione superiore. L'obiettivo duale dell'SVM dipende dai campioni di addestramento solo attraverso i prodotti scalari, e possiamo calcolare un punteggio di classificazione attraverso prodotti scalari tra campioni di addestramento e valutazione. Pertanto, l'SVM non richiede che calcoliamo esplicitamente l'espansione delle caratteristiche: è sufficiente che siamo in grado di calcolare il prodotto scalare tra le caratteristiche espanse k(x₁, x₂) = φ(x₁)ᵀφ(x₂). La funzione k è chiamata funzione kernel.

Implementare il classificatore SVM duale (senza bias) per funzioni kernel generiche. È possibile riutilizzare il codice sviluppato in precedenza, tuttavia è necessario sostituire il calcolo di H_c con:

$$H_c^{i,j} = z_i z_j k(x_i, x_j)$$

Diversamente dall'SVM lineare, non siamo in grado di calcolare direttamente la soluzione primale e il suo costo. Tuttavia, possiamo sfruttare la relazione di soluzione primale-duale per esprimere l'obiettivo primale in termini di α:

$$J_b(w_b) = \frac{1}{2}\|w_b\|^2 + C\sum_{i=1}^{n}\max(0, 1 - z_i(w_b^T x_b^i)) = \frac{1}{2}\alpha^T H \alpha + C\sum_{i=1}^{n}\max(0, 1 - H_{i\cdot}\alpha)$$

dove H_{i·} è la i-esima riga di H. Si noti che i termini H_{i·}α corrispondono alle componenti del vettore Hα, e possono quindi essere calcolati da quest'ultimo.

Il punteggio di un campione di test può essere calcolato come:

$$s(x_t) = \sum_{i=1}^{n}\alpha_i^* z_i k(x_i, x_t)$$

dove la sommatoria è estesa ai campioni di addestramento (in pratica possiamo considerare solo i campioni per i quali α_i > 0, cioè i vettori di supporto).

È possibile provare diversi kernel:

- **Kernel polinomiale di grado d**: $k(x_1, x_2) = (x_1^T x_2 + c)^d$
- **Kernel Radial Basis Function**: $k(x_1, x_2) = e^{-\gamma\|x_1 - x_2\|^2}$

La scelta del kernel e dei suoi iper-parametri (ad es. c e γ) può essere effettuata anche tramite validazione incrociata.

**NOTA**: Per aggiungere un bias (regolarizzato) nella versione SVM non-lineare non è sufficiente estendere semplicemente i vettori di caratteristiche come fatto in precedenza, ma dovremmo invece aggiungere un valore costante alla nostra funzione kernel:

$$\tilde{k}(x_1, x_2) = k(x_1, x_2) + \xi$$

In [125]:
def polygrad_d(d,DVAL,c):
    return (DTR.T @ DVAL + c) ** d
def KRB(D1, D2, gamma):
    # Calcolo delle norme al quadrato delle colonne per il primo dataset
    # Il reshape(-1, 1) assicura che il risultato sia un vettore colonna
    D1_norm2 = (D1 ** 2).sum(axis=0).reshape(-1, 1)
    
    # Calcolo delle norme al quadrato delle colonne per il secondo dataset
    # Il reshape(1, -1) assicura che il risultato sia un vettore riga
    D2_norm2 = (D2 ** 2).sum(axis=0).reshape(1, -1)
    
    # Calcolo della matrice delle distanze al quadrato sfruttando il broadcasting
    dist2 = D1_norm2 + D2_norm2 - 2 * (D1.T @ D2)
    
    # Applicazione dell'esponenziale scalato per gamma
    return np.exp(-gamma * dist2)

def regular_kernel(kernel,xi):
    return kernel+xi

def train_kernel_poly_SVM(K, C, DTR, LTR,xi,d,c,):
    N = DTR.shape[1]
    ZTR = LTR * 2 - 1
    
    
    kernel_reg=regular_kernel(polygrad_d(d,DTR,c),xi)
    
    H = getCol(ZTR) * getRow(ZTR) * kernel_reg

    def fOpt(alpha):
        Ha = H @ alpha
        loss = 0.5 * alpha.T @ Ha - alpha.sum()
        grad = Ha - np.ones(alpha.size)
        return loss, grad

    x0 = np.zeros(N)
    alpha, _, _ = scipy.optimize.fmin_l_bfgs_b(
        func=fOpt,
        x0=x0,
        approx_grad=False,
        bounds=[(0, C) for _ in range(N)],
        factr=np.nan,
        pgtol=1e-5
    )
    
    
    dual_loss,_=fOpt(alpha)
    primal_loss=0.5*alpha.T @(H@alpha)+C*np.maximum(0,1-H@alpha).sum()
    print(f"Primal Loss : {primal_loss} - Dual Loss : {-dual_loss}")
    return alpha


def train_kernel_KRB(K, C, DTR, LTR,xi,gamma):
    N = DTR.shape[1]
    ZTR = LTR * 2 - 1
    
    
    kernel_reg=regular_kernel(KRB(DTR,DTR,gamma),xi)
    
    H = getCol(ZTR) * getRow(ZTR) * kernel_reg

    def fOpt(alpha):
        Ha = H @ alpha
        loss = 0.5 * alpha.T @ Ha - alpha.sum()
        grad = Ha - np.ones(alpha.size)
        return loss, grad

    x0 = np.zeros(N)
    alpha, _, _ = scipy.optimize.fmin_l_bfgs_b(
        func=fOpt,
        x0=x0,
        approx_grad=False,
        bounds=[(0, C) for _ in range(N)],
        factr=np.nan,
        pgtol=1e-5
    )
    
    
    dual_loss,_=fOpt(alpha)
    primal_loss=0.5*alpha.T @(H@alpha)+C*np.maximum(0,1-H@alpha).sum()
    print(f"Primal Loss : {primal_loss} - Dual Loss : {-dual_loss}")
    return alpha

In [128]:

KC_poly = np.array([
    [0.0, 1.0,2,0],
    [1.0, 1.0,2,0],
    [0.0, 1.0,2,1],
    [1.0,1.0,2,1],
])
KC_KRB = np.array([
    [0.0, 1.0,1],
    [1.0, 1.0,1],
    [0.0, 1.0,10],
    [1.0,1.0,10],
])
pi_T=0.5
ZTR=LTR*2-1
for kc in KC_poly:
    K=kc[0]
    C=kc[1]
    d=kc[2]
    c=kc[3]
    xi=K**2

    print(f"================================= K: {K} C: {C} Poly (d={d},c={c})==============================================================")
    alpha=train_kernel_poly_SVM(K,C,DTR,LTR,xi,d,c)
    reg_kernel=regular_kernel(polygrad_d(d,DVAL,c),xi)
    s=alpha*ZTR @ reg_kernel
    predictions=s >0
    error_rate=np.mean(predictions != LVAL)
    min_dcf=compute_min_dcf(s,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(s, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)


    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

    print(f"==================================================================================================")



================================= K: 0.0 C: 1.0 Poly (d=2.0,c=0.0)==============================================================
Primal Loss : 6.66366518392721 - Dual Loss : 6.66362783094242
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 1.0 C: 1.0 Poly (d=2.0,c=0.0)==============================================================
Primal Loss : 6.296934230180396 - Dual Loss : 6.296912183696072
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 0.0 C: 1.0 Poly (d=2.0,c=1.0)==============================================================
Primal Loss : 3.5929321962289453 - Dual Loss : 3.592918442263641
Error Rate: 2.9%
DCF min: 0.05555555555555555
Actual DCF:0.05555555555555555

================================= K: 1.0 C: 1.0 Poly (d=2.0,c=1.0)==============================================================
Primal Loss : 3.5699944542349797 - Dual Loss : 3.569986579777602
Error Rate: 2.9%
DCF m

In [127]:

for kc in KC_KRB:
    K=kc[0]
    C=kc[1]
    gamma=kc[2]
    xi=K**2

    print(f"================================= K: {K} C: {C} KRB (d={d},c={c})==============================================================")
    alpha=train_kernel_KRB(K,C,DTR,LTR,xi,gamma)
    reg_kernel=regular_kernel(KRB(DTR,DVAL,gamma),xi)
    s=alpha*ZTR @ reg_kernel
    predictions=s >0
    error_rate=np.mean(predictions != LVAL)
    min_dcf=compute_min_dcf(s,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(s, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)


    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

    print(f"==================================================================================================")

================================= K: 0.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 11.989317276783288 - Dual Loss : 11.98929841469204
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 1.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 11.978144832456012 - Dual Loss : 11.978133312353526
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 0.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 18.42762364791104 - Dual Loss : 18.42756867178749
Error Rate: 11.8%
DCF min: 0.1736111111111111
Actual DCF:0.22916666666666666

================================= K: 1.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 18.391862918156956 - Dual Loss : 18.39181622127951
Error Rate: 8.8%
DCF m